This Colab notebook includes all the code you need to experiment with page-level document classification using a RAG-based strategy. Upload your own multi-document pharmaceutical PDF, follow along step-by-step, and learn how to detect document boundaries, label each section by type, and generate clean metadata for smarter extraction and automation.


**Step 1: Extract Page-Level Content from Pharmaceutical PDF**





In [7]:
# ============================================
# STEP 1: Install Required Libraries
# ============================================
#
# WHAT WE'RE DOING:
# Installing PyPDF2, a Python library for reading and extracting
# text from PDF files.
#
# WHY THIS MATTERS:
# Pharmaceutical bundled files are multi-document PDFs. We need a
# reliable way to extract text from each page before we can
# classify or route documents.
#
# WHAT YOU'LL SEE:
# A brief installation log confirming PyPDF2 was installed.
# ============================================

!pip install -q PyPDF2

In [8]:
# ============================================
# STEP 2: Extract Page-Level Content from PDF
# ============================================
#
# WHAT WE'RE DOING:
# Loading the pharmaceutical blob PDF and extracting the raw
# text from every page into a list of dictionaries.
#
# WHY THIS MATTERS:
# Each page in a pharma blob file may belong to a different
# document (Certificate of Quality, BSE/TSE Declaration, etc.).
# Extracting per-page text is the foundation for classification.
#
# WHAT YOU'LL SEE:
# A list of dictionaries, each containing a page number and
# the extracted text content for that page.
# ============================================

from PyPDF2 import PdfReader

reader = PdfReader("/content/pharma-blob-sample.pdf")
pages = [page.extract_text() for page in reader.pages]
doc_pages = [{"page_num": i, "text": p} for i, p in enumerate(pages)]
doc_pages

[{'page_num': 0,
  'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752\nUnited States\nPage 1 / 1\ncytiva.com\n3 June, 2022\nRe: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating\nInstructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits\nas well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to\nbrittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to\n+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of\ndamage to the kit during setup and handling.\nDescription Part Number Operating Temperature\nHigh Flow Kit F, Modified, AKTA ready 29477427 +2°C to +40°C\nHigh Flow Gradient C, Modified, AKTA ready 29184612 +2°C to +4

**Step 2: Set Up Mistral LLM Connection**

In [3]:
# Install required libraries with CUDA support
!pip install -q torch

In [4]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


In [5]:
# Check CUDA version first
!nvcc --version

# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu123
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.5/444.5 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 100.2 MB/s eta 0:00:00


In [7]:
!pip install llama-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, w

In [1]:
from llama_cpp import Llama
import os

# Download Mistral model if not already present
model_path = "/content/mistral.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

Model file exists. Size: 4166.07 MB


llm_load_print_meta: n_ctx_orig_yarn  = 32768
llm_load_print_meta: rope_finetuned   = unknown
llm_load_print_meta: ssm_d_conv       = 0
llm_load_print_meta: ssm_d_inner      = 0
llm_load_print_meta: ssm_d_state      = 0
llm_load_print_meta: ssm_dt_rank      = 0
llm_load_print_meta: ssm_dt_b_c_rms   = 0
llm_load_print_meta: model type       = 7B
llm_load_print_meta: model ftype      = Q4_K - Medium
llm_load_print_meta: model params     = 7.24 B
llm_load_print_meta: model size       = 4.07 GiB (4.83 BPW) 
llm_load_print_meta: general.name     = mistralai_mistral-7b-instruct-v0.2
llm_load_print_meta: BOS token        = 1 '<s>'
llm_load_print_meta: EOS token        = 2 '</s>'
llm_load_print_meta: UNK token        = 0 '<unk>'
llm_load_print_meta: PAD token        = 0 '<unk>'
llm_load_print_meta: LF token         = 13 '<0x0A>'
llm_load_print_meta: max token length = 48
ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:

Model loaded successfully!


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | AVX512_BF16 = 0 | FMA = 1 | NEON = 0 | SVE = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | MATMUL_INT8 = 0 | LLAMAFILE = 1 | 
Model metadata: {'tokenizer.chat_template': "{{ bos_token }}{% for message in messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}{% elif message['role'] == 'assistant' %}{{ message['content'] + eos_token}}{% else %}{{ raise_exception('Only user and assistant roles are supported!') }}{% endif %}{% endfor %}", 'tokenizer.ggml.add_eos_token': 'false', 'tokenizer.ggml.padding_token_id': '0', 'tokenizer.ggml.unknown_token_id': '0', 'tokenizer.ggml.eos_token_id': '2', 'general.architecture': 'llama', 'llama.rope.freq_base': 

### Create a local Mistral helper function

In [2]:
def local_mistral_model(prompt, max_tokens=120, temperature=0.0):
    """
    Sends a prompt to the locally loaded Mistral model using llama-cpp.
    This replaces the Gemini API call.
    """

    formatted_prompt = f"""
[INST]
{prompt}
[/INST]
"""

    response = llm(
        formatted_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=["</s>", "[/INST]"]
    )

    return response["choices"][0]["text"].strip()

### Add a helper to keep prompts short

In [3]:
def trim_text(text, max_chars=2500):
    """
    Keeps the page text short enough for the local model context window.
    """
    if len(text) <= max_chars:
        return text

    return text[:max_chars]

### Replace the Gemini-based boundary detection function

In [4]:
def is_same_document(prev_text, curr_text, doc_type=None):
    prev_text = trim_text(prev_text, max_chars=1800)
    curr_text = trim_text(curr_text, max_chars=1800)

    prompt = f"""
You are checking whether two consecutive pages from a pharmaceutical
document package belong to the SAME individual document.

A NEW document starts when the page has:
- A different document title or heading
- A different topic or subject matter
- Its own header with a new document number or reference
- A new product, lot number, or certificate section

Pages belong to the SAME document when:
- The second page continues the previous page
- They share the same document title, document number, or subject
- The second page says continued or page 2 of 2

Previous page type: {doc_type or 'unknown'}

Previous Page:
{prev_text}

Current Page:
{curr_text}

Do these two pages belong to the SAME document?

Answer ONLY one word: Yes or No.
"""

    response = local_mistral_model(prompt, max_tokens=20)

    print("Boundary response:", response)

    return response.strip().lower().startswith("yes")

### Replace the Gemini-based document classifier

In [5]:
def classify_document_type(text):
    text = trim_text(text, max_chars=3000)

    prompt = f"""
You are a pharmaceutical document classifier.

Classify the page content into ONE of these document types:

- Cover Letter
- Certificate of Quality
- Packaging Specification
- BSE/TSE Declaration
- Material Description
- Supplier Qualification
- Chain of Custody
- Other

Use these rules:
- Cover Letter: formal letter, often starts with "To Whom It May Concern"
- Certificate of Quality: contains lot numbers, manufacture dates, expiration dates, and test results
- Packaging Specification: describes packaging components, materials, part numbers, and change history
- BSE/TSE Declaration: discusses animal-origin materials or TSE/BSE compliance
- Material Description: lists materials of construction or physical properties
- Supplier Qualification: contains audit history, certifications, or approved supplier information
- Chain of Custody: contains traceability or manufacturing-to-shipment flow

Page Content:
{text}

Respond with ONLY the document type name.
"""

    response = local_mistral_model(prompt, max_tokens=40)

    cleaned_response = response.strip().replace(".", "")

    valid_types = [
        "Cover Letter",
        "Certificate of Quality",
        "Packaging Specification",
        "BSE/TSE Declaration",
        "Material Description",
        "Supplier Qualification",
        "Chain of Custody",
        "Other"
    ]

    for doc_type in valid_types:
        if doc_type.lower() in cleaned_response.lower():
            return doc_type

    return "Other"

### Test the open-source classifier on one page

In [9]:
classify_document_type(doc_pages[0]["text"])


llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.18 ms /     4 runs   (    0.05 ms per token, 21978.02 tokens per second)
llama_print_timings: prompt eval time =    3218.53 ms /   629 tokens (    5.12 ms per token,   195.43 tokens per second)
llama_print_timings:        eval time =    1779.27 ms /     3 runs   (  593.09 ms per token,     1.69 tokens per second)
llama_print_timings:       total time =    5001.78 ms /   632 tokens


'Cover Letter'

### Test document boundary detection

In [10]:
prev_text = doc_pages[1]["text"]
curr_text = doc_pages[2]["text"]
doc_type = "Certificate of Quality"

is_same_document(prev_text, curr_text, doc_type)

Llama.generate: 10 prefix-match hit, remaining 711 prompt tokens to eval

llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.90 ms /    20 runs   (    0.05 ms per token, 22172.95 tokens per second)
llama_print_timings: prompt eval time =    3373.19 ms /   711 tokens (    4.74 ms per token,   210.78 tokens per second)
llama_print_timings:        eval time =   11743.05 ms /    19 runs   (  618.06 ms per token,     1.62 tokens per second)
llama_print_timings:       total time =   15134.22 ms /   730 tokens


Boundary response: No.

Explanation: The previous page is for the "AKTA ready Low Flow


False

### Rerun the full segmentation pipeline using Mistral

In [11]:
results = []
current_doc_type = None
doc_counter = 0

for i, page in enumerate(doc_pages):
    page_text = page["text"]

    if i == 0:
        current_doc_type = classify_document_type(page_text)
    else:
        prev_text = doc_pages[i - 1]["text"]

        same = is_same_document(prev_text, page_text, current_doc_type)

        if not same:
            doc_counter += 1
            current_doc_type = classify_document_type(page_text)

    results.append({
        "page": i + 1,
        "doc_id": doc_counter,
        "doc_type": current_doc_type
    })

for r in results:
    print(r)

Llama.generate: 10 prefix-match hit, remaining 619 prompt tokens to eval

llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.19 ms /     4 runs   (    0.05 ms per token, 20725.39 tokens per second)
llama_print_timings: prompt eval time =    3263.42 ms /   619 tokens (    5.27 ms per token,   189.68 tokens per second)
llama_print_timings:        eval time =    1831.95 ms /     3 runs   (  610.65 ms per token,     1.64 tokens per second)
llama_print_timings:       total time =    5098.26 ms /   622 tokens
Llama.generate: 10 prefix-match hit, remaining 827 prompt tokens to eval

llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.91 ms /    20 runs   (    0.05 ms per token, 21978.02 tokens per second)
llama_print_timings: prompt eval time =    2788.80 ms /   827 tokens (    3.37 ms per token,   296.54 tokens per second)
llama_print_timings:        eval time =   12131.90 ms /    19 runs   (

Boundary response: Yes.

Both pages are related to the same product, AKTA ready, and



llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.94 ms /    20 runs   (    0.05 ms per token, 21186.44 tokens per second)
llama_print_timings: prompt eval time =    2494.95 ms /   575 tokens (    4.34 ms per token,   230.47 tokens per second)
llama_print_timings:        eval time =   12033.80 ms /    19 runs   (  633.36 ms per token,     1.58 tokens per second)
llama_print_timings:       total time =   14543.64 ms /   594 tokens
Llama.generate: 10 prefix-match hit, remaining 501 prompt tokens to eval


Boundary response: No.

Explanation:
The previous page is for the "AKTA ready Low



llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.18 ms /     4 runs   (    0.04 ms per token, 22471.91 tokens per second)
llama_print_timings: prompt eval time =    1491.19 ms /   501 tokens (    2.98 ms per token,   335.97 tokens per second)
llama_print_timings:        eval time =    1783.09 ms /     3 runs   (  594.36 ms per token,     1.68 tokens per second)
llama_print_timings:       total time =    3278.31 ms /   504 tokens
Llama.generate: 10 prefix-match hit, remaining 716 prompt tokens to eval

llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.88 ms /    20 runs   (    0.04 ms per token, 22831.05 tokens per second)
llama_print_timings: prompt eval time =    3341.36 ms /   716 tokens (    4.67 ms per token,   214.28 tokens per second)
llama_print_timings:        eval time =   11839.96 ms /    19 runs   (  623.16 ms per token,     1.60 tokens per second)
llama_print_timings:  

...

Boundary response: Based on the information provided, the answer is: Yes.

Both pages appear to be



llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.23 ms /     5 runs   (    0.05 ms per token, 21459.23 tokens per second)
llama_print_timings: prompt eval time =    2901.41 ms /   614 tokens (    4.73 ms per token,   211.62 tokens per second)
llama_print_timings:        eval time =    2373.49 ms /     4 runs   (  593.37 ms per token,     1.69 tokens per second)
llama_print_timings:       total time =    5279.33 ms /   618 tokens
Llama.generate: 10 prefix-match hit, remaining 904 prompt tokens to eval

llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.99 ms /    20 runs   (    0.05 ms per token, 20181.63 tokens per second)
llama_print_timings: prompt eval time =    2915.94 ms /   904 tokens (    3.23 ms per token,   310.02 tokens per second)
llama_print_timings:        eval time =   12206.88 ms /    19 runs   (  642.47 ms per token,     1.56 tokens per second)
llama_print_timings:  

Boundary response: Yes.

Explanation: The current page indicates that it is a continuation of the



llama_print_timings:        load time =    2184.63 ms
llama_print_timings:      sample time =       0.97 ms /    20 runs   (    0.05 ms per token, 20682.52 tokens per second)
llama_print_timings: prompt eval time =    2620.99 ms /   697 tokens (    3.76 ms per token,   265.93 tokens per second)
llama_print_timings:        eval time =   11929.92 ms /    19 runs   (  627.89 ms per token,     1.59 tokens per second)
llama_print_timings:       total time =   14565.33 ms /   716 tokens


Boundary response: Yes.

Both pages are related to the supplier qualification record of Cytiva Sweden
{'page': 1, 'doc_id': 0, 'doc_type': 'Cover Letter'}
{'page': 2, 'doc_id': 0, 'doc_type': 'Cover Letter'}
{'page': 3, 'doc_id': 1, 'doc_type': 'Certificate of Quality'}
{'page': 4, 'doc_id': 1, 'doc_type': 'Certificate of Quality'}
{'page': 5, 'doc_id': 1, 'doc_type': 'Certificate of Quality'}
{'page': 6, 'doc_id': 1, 'doc_type': 'Certificate of Quality'}
{'page': 7, 'doc_id': 1, 'doc_type': 'Certificate of Quality'}
{'page': 8, 'doc_id': 2, 'doc_type': 'Supplier Qualification'}
{'page': 9, 'doc_id': 2, 'doc_type': 'Supplier Qualification'}
{'page': 10, 'doc_id': 2, 'doc_type': 'Supplier Qualification'}


### Convert results to a DataFrame

In [12]:
import pandas as pd

df = pd.DataFrame(results)
df

,page,doc_id,doc_type
0,1,0,Cover Letter
1,2,0,Cover Letter
2,3,1,Certificate of Quality
3,4,1,Certificate of Quality
4,5,1,Certificate of Quality
5,6,1,Certificate of Quality
6,7,1,Certificate of Quality
7,8,2,Supplier Qualification
8,9,2,Supplier Qualification
9,10,2,Supplier Qualification
